In [1]:
import pandas as pd
import duckdb
import pyarrow

print("pandas:", pd.__version__)
print("duckdb:", duckdb.__version__)
print("pyarrow:", pyarrow.__version__)

pandas: 3.0.5
duckdb: 1.5.5
pyarrow: 25.0.1


In [2]:
import duckdb

query = """
SELECT *
FROM read_csv_auto('../data/raw/2019-Oct.csv')
LIMIT 5
"""

df_sample = duckdb.sql(query).df()
df_sample

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [3]:
query = """
SELECT COUNT(*) AS total_rows
FROM read_csv_auto('../data/raw/2019-Oct.csv')
"""

duckdb.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,42448764


查看用户行为类型

In [4]:
query = """
SELECT
    event_type,
    COUNT(*) AS event_count
FROM read_csv_auto('../data/raw/2019-Oct.csv')
GROUP BY event_type
ORDER BY event_count DESC
"""

event_counts = duckdb.sql(query).df()
event_counts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,event_type,event_count
0,view,40779399
1,cart,926516
2,purchase,742849


查看字段类型

In [5]:
query = """
DESCRIBE
SELECT *
FROM read_csv_auto('../data/raw/2019-Oct.csv')
"""

column_info = duckdb.sql(query).df()
column_info

,column_name,column_type,null,key,default,extra
0,event_time,TIMESTAMP,YES,None,None,None
1,event_type,VARCHAR,YES,None,None,None
2,product_id,BIGINT,YES,None,None,None
3,category_id,BIGINT,YES,None,None,None
4,category_code,VARCHAR,YES,None,None,None
5,brand,VARCHAR,YES,None,None,None
6,price,DOUBLE,YES,None,None,None
7,user_id,BIGINT,YES,None,None,None
8,user_session,VARCHAR,YES,None,None,None


检查缺失值

In [6]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(event_time) AS event_time_missing,
    COUNT(*) - COUNT(event_type) AS event_type_missing,
    COUNT(*) - COUNT(product_id) AS product_id_missing,
    COUNT(*) - COUNT(category_id) AS category_id_missing,
    COUNT(*) - COUNT(category_code) AS category_code_missing,
    COUNT(*) - COUNT(brand) AS brand_missing,
    COUNT(*) - COUNT(price) AS price_missing,
    COUNT(*) - COUNT(user_id) AS user_id_missing,
    COUNT(*) - COUNT(user_session) AS user_session_missing
FROM read_csv_auto('../data/raw/2019-Oct.csv')
"""

missing_values = duckdb.sql(query).df()
missing_values

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,event_time_missing,event_type_missing,product_id_missing,category_id_missing,category_code_missing,brand_missing,price_missing,user_id_missing,user_session_missing
0,42448764,0,0,0,0,13515609,6113008,0,0,2


In [7]:
query = """
SELECT
    MIN(price) AS min_price,
    MAX(price) AS max_price,
    AVG(price) AS avg_price,
    SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) AS non_positive_price
FROM read_csv_auto('../data/raw/2019-Oct.csv')
"""

price_check = duckdb.sql(query).df()
price_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_price,max_price,avg_price,non_positive_price
0,0.0,2574.07,290.323661,68673.0


检查时间范围和核心规模

In [8]:
query = """
SELECT
    MIN(event_time) AS start_time,
    MAX(event_time) AS end_time,
    COUNT(DISTINCT user_id) AS user_count,
    COUNT(DISTINCT product_id) AS product_count,
    COUNT(DISTINCT category_id) AS category_count,
    COUNT(DISTINCT brand) AS brand_count
FROM read_csv_auto('../data/raw/2019-Oct.csv')
"""

basic_stats = duckdb.sql(query).df()
basic_stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_time,end_time,user_count,product_count,category_count,brand_count
0,2019-10-01,2019-10-31 23:59:59,3022290,166794,624,3445


# 01 数据概览

本部分主要对 2019 年 10 月电商用户行为数据进行初步检查，包括数据规模、字段结构、用户行为类型、缺失值、价格范围以及用户和商品规模，为后续数据清洗和业务分析做准备。

由于原始 CSV 数据量较大，本项目使用 DuckDB 直接查询原始文件，避免一次性将全部数据加载到内存。

## 数据概览小结

2019 年 10 月数据共包含 42,448,764 条用户行为记录，覆盖 3,022,290 名用户和 166,794 个商品，行为类型包括浏览、加购和购买。

数据核心字段整体较完整，主要缺失集中在 category_code 和 brand 字段，同时存在部分价格为 0 的记录。后续将根据不同分析场景进行针对性处理，并将原始 CSV 转换为更适合分析的 Parquet 格式。